# Phần B1 — Đánh giá BM25 trên SciFact

## Mục tiêu
- Tải và đọc bộ dữ liệu SciFact từ BEIR.
- Xây dựng chỉ mục nghịch đảo bằng BM25 tự viết.
- Thử nghiệm tìm kiếm trên một truy vấn.
- Chạy BM25 trên toàn bộ 300 truy vấn.
- Chuẩn bị kết quả để đánh giá bằng các chỉ số IR.

**Cấu hình:** k1 = 1.2, b = 0.75

**Tokenizer:** `tach_khoang_trang` từ `bm25.py`.

In [1]:
import sys
from pathlib import Path

from beir import util
from beir.datasets.data_loader import GenericDataLoader

# Notebook đang chạy trong phanB
PHAN_B_DIR = Path.cwd().resolve()

# Thư mục gốc chứa bm25.py
ROOT_DIR = PHAN_B_DIR.parent

# Dữ liệu nằm trong phanB/du_lieu
DATA_DIR = PHAN_B_DIR / "du_lieu"

sys.path.insert(0, str(ROOT_DIR))

from bm25 import ChiMuc

print("Project root:", ROOT_DIR)
print("Data directory:", DATA_DIR)

Project root: D:\RAG\tep_kem
Data directory: D:\RAG\tep_kem\phanB\du_lieu


d:\RAG\tep_kem\.venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
# url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

# dataset_path = util.download_and_unzip(url, str(DATA_DIR))

from collections import Counter

dataset_path = DATA_DIR / "scifact"

corpus, queries, qrels = GenericDataLoader(
    data_folder=str(dataset_path)
).load(split="test")

phan_bo_nhan = Counter()

for nhan_truy_van in qrels.values():
    phan_bo_nhan.update(nhan_truy_van.values())

print("Phân bố nhãn liên quan trong SciFact:")

for muc_nhan, so_luong in sorted(phan_bo_nhan.items()):
    print(f"Nhãn {muc_nhan}: {so_luong}")

print("Tổng số phán định:", sum(phan_bo_nhan.values()))
print("Số tài liệu:", len(corpus))
print("Số truy vấn:", len(queries))
print("Số truy vấn có nhãn:", len(qrels))

100%|██████████| 5183/5183 [00:00<00:00, 206550.98it/s]

Phân bố nhãn liên quan trong SciFact:
Nhãn 1: 339
Tổng số phán định: 339
Số tài liệu: 5183
Số truy vấn: 300
Số truy vấn có nhãn: 300



### 1. Cấu trúc bộ dữ liệu SciFact

Bộ dữ liệu **SciFact** thuộc bộ benchmark **BEIR**, gồm ba thành phần chính:

| Thành phần | File | Vai trò |
|---|---|---|
| **Corpus** | `corpus.jsonl` | Kho tài liệu để BM25 tìm kiếm |
| **Queries** | `queries.jsonl` | Các truy vấn đầu vào |
| **Qrels** | `qrels/test.tsv` | Nhãn liên quan dùng để đánh giá kết quả truy xuất |

**Thống kê tập test:**
- **5.183** tài liệu trong `corpus`.
- **300** truy vấn trong `queries`.
- **300** truy vấn có nhãn liên quan trong `qrels`.

**Quy trình:** BM25 sử dụng `corpus` để xây dựng chỉ mục và `queries` để tìm kiếm. Sau đó, kết quả truy xuất được đối chiếu với `qrels` để tính các chỉ số đánh giá như nDCG@10, Recall@100, MRR và MAP.


In [18]:
dataset_path = DATA_DIR / "scifact"

corpus, queries, qrels = GenericDataLoader(
    data_folder=str(dataset_path)
).load(split="test")

print("Số tài liệu:", len(corpus))
print("Số truy vấn:", len(queries))
print("Số truy vấn có nhãn:", len(qrels))

100%|██████████| 5183/5183 [00:00<00:00, 55236.87it/s]

Số tài liệu: 5183
Số truy vấn: 300
Số truy vấn có nhãn: 300


In [29]:
from pprint import pprint

corpus_iter = iter(corpus.items())
queries_iter = iter(queries.items())
qrels_iter = iter(qrels.items())

# Lấy phần tử đầu tiên
doc_id, doc = next(corpus_iter)
query_id, query_text = next(queries_iter)
qrel_query_id, relevance = next(qrels_iter)


print("=== 1. CORPUS ===")
print("Document ID:", doc_id)
pprint(doc)

print("\n=== 2. QUERIES ===")
print("Query ID:", query_id)
print("Query text:", query_text)

print("\n=== 3. QRELS ===")
print("Query ID:", qrel_query_id)
pprint(relevance)

=== 1. CORPUS ===
Document ID: 4983
{'text': 'Alterations of the architecture of cerebral white matter in the '
         'developing human brain can affect cortical development and result in '
         'functional disabilities. A line scan diffusion-weighted magnetic '
         'resonance imaging (MRI) sequence with diffusion tensor analysis was '
         'applied to measure the apparent diffusion coefficient, to calculate '
         'relative anisotropy, and to delineate three-dimensional fiber '
         'architecture in cerebral white matter in preterm (n = 17) and '
         'full-term infants (n = 7). To assess effects of prematurity on '
         'cerebral white matter development, early gestation preterm infants '
         '(n = 10) were studied a second time at term. In the central white '
         'matter the mean apparent diffusion coefficient at 28 wk was high, '
         '1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the '
         'posterior limb of the 


### 2. Khám phá cấu trúc dữ liệu SciFact

Sau khi tải dữ liệu bằng `GenericDataLoader`, ba thành phần được biểu diễn dưới dạng dictionary trong Python:

**1. Corpus**

- Khóa (`doc_id`): Mã định danh tài liệu.
- Giá trị (`doc`): Dictionary chứa các trường như `title` và `text`.
- Được sử dụng để xây dựng chỉ mục nghịch đảo phục vụ tìm kiếm BM25.

**2. Queries**

- Khóa (`query_id`): Mã định danh truy vấn.
- Giá trị (`query_text`): Nội dung truy vấn dạng chuỗi.
- Được đưa vào mô hình BM25 để tìm kiếm tài liệu.

**3. Qrels**

- Khóa (`query_id`): Mã định danh truy vấn.
- Giá trị: Dictionary ánh xạ `doc_id` sang mức độ liên quan (`relevance`).
- Được sử dụng làm nhãn chuẩn để đánh giá kết quả truy xuất, không tham gia vào quá trình tính điểm BM25.

**Lưu ý:** `query_id` liên kết dữ liệu trong `queries` với nhãn tương ứng trong `qrels`; `doc_id` liên kết kết quả truy xuất với tài liệu trong `corpus`.


## Xây dựng chỉ mục BM25

In [20]:
kho = {
    doc_id: doc['title'] + ' ' + doc['text'] for doc_id, doc in corpus.items()
}

chi_muc = ChiMuc(kho)

print("Tổng số tài liệu:", chi_muc.n)
print("Độ dài trung bình:", chi_muc.avgdl)
print("Số token khác nhau:", len(chi_muc.chi_muc))

Tổng số tài liệu: 5183
Độ dài trung bình: 214.6280146633224
Số token khác nhau: 87379


In [21]:

# Lấy truy vấn đầu tiên có nhãn trong qrels
iterable = iter(qrels)
query_id = next(iterable)
query_text = queries[query_id]

# Tìm 10 tài liệu có điểm BM25 cao nhất
ket_qua = chi_muc.tim(query_text, k=10, k1=1.2, b=0.75)

print("Query ID:", query_id)
print("Query:", query_text)
print("Relevant documents:", qrels[query_id])

print("\nTop 10 BM25:")
for rank, (doc_id, score) in enumerate(ket_qua, start=1):
    print(f"{rank}. {doc_id}: {score:.4f}")


Query ID: 1
Query: 0-dimensional biomaterials show inductive properties.
Relevant documents: {'31715818': 1}

Top 10 BM25:
1. 825728: 9.3064
2. 10931595: 9.1688
3. 43385013: 7.6249
4. 13231899: 7.4993
5. 18953920: 7.2009
6. 45638119: 7.0799
7. 17518195: 7.0113
8. 24998637: 6.9319
9. 11674596: 6.8081
10. 12824568: 6.6601



### 3. Thử nghiệm truy xuất bằng BM25

Sau khi xây dựng chỉ mục nghịch đảo, hệ thống thực hiện truy xuất thử nghiệm trên một truy vấn thuộc tập test SciFact.

**Cấu hình BM25:**
- `k1 = 1.2`: Tham số điều chỉnh ảnh hưởng của tần suất xuất hiện từ.
- `b = 0.75`: Tham số điều chỉnh chuẩn hóa độ dài tài liệu.
- `k = 10`: Trả về tối đa 10 tài liệu có điểm BM25 cao nhất.

**Quy trình:**
1. Lấy một truy vấn có nhãn từ `qrels`.
2. Truy xuất nội dung tương ứng trong `queries`.
3. Tính điểm BM25 và xếp hạng tài liệu.
4. Đối chiếu danh sách tài liệu được truy xuất với nhãn liên quan trong `qrels`.

**Mục đích:** Kiểm tra hoạt động của hệ thống truy xuất trước khi đánh giá trên toàn bộ tập test.


## Truy xuất toàn bộ SciFact

In [33]:

# Lưu kết quả BM25 cho toàn bộ truy vấn
ket_qua_bm25 = {}

for query_id, query_text in queries.items():
    # Lấy tối đa 100 tài liệu có điểm BM25 cao nhất
    danh_sach = chi_muc.tim(query_text, k=100, k1=1.2, b=0.75)
    '''
                danh_sach = [
                ("825728", 9.3064),
                ("10931595", 9.1688),
                ("43385013", 7.6249)
            ]
    '''
    # Chuyển danh sách tuple thành dictionary
    ket_qua_bm25[query_id] = {
        doc_id: score
        for doc_id, score in danh_sach
    }
    '''
        {
            "query_id": {
                "doc_id": score
            }
        }
    '''

print("Số truy vấn đã xử lý:", len(ket_qua_bm25))

# Kiểm tra kết quả của một truy vấn
query_id = next(iter(ket_qua_bm25))

print("Query ID:", query_id)
print("Số tài liệu truy xuất:", len(ket_qua_bm25[query_id]))
print("Rank", "Doc ID", "BM25 Score", sep="\t")

for rank, (doc_id, score) in enumerate(
    list(ket_qua_bm25[query_id].items())[:5], start=1
):
    print(rank, doc_id, round(score, 4), sep="\t")

Số truy vấn đã xử lý: 300
Query ID: 1
Số tài liệu truy xuất: 100
Rank	Doc ID	BM25 Score
1	825728	9.3064
2	10931595	9.1688
3	43385013	7.6249
4	13231899	7.4993
5	18953920	7.2009



### 4. Truy xuất BM25 trên toàn bộ tập test SciFact

Sau khi kiểm tra truy xuất trên một truy vấn mẫu, hệ thống tiến hành chạy BM25 trên toàn bộ 300 truy vấn của tập test SciFact.

**Thiết lập thí nghiệm:**
- Số tài liệu: 5.183.
- Số truy vấn: 300.
- Tham số BM25: `k1 = 1.2`, `b = 0.75`.
- Số tài liệu truy xuất tối đa cho mỗi truy vấn: `k = 100`.
- Phương pháp tách từ: chuyển chữ thường và tách theo khoảng trắng.

Kết quả được lưu dưới dạng dictionary lồng nhau:

`{query_id: {doc_id: bm25_score}}`

Cấu trúc này cho phép đối chiếu kết quả truy xuất với nhãn liên quan (`qrels`) để tính các chỉ số đánh giá.



### 5. Đánh giá chất lượng truy xuất BM25

Sau khi truy xuất toàn bộ 300 truy vấn SciFact, kết quả BM25 được đối chiếu với nhãn liên quan (`qrels`) để đánh giá chất lượng tìm kiếm.

Các chỉ số sử dụng:
- **nDCG@10:** Đánh giá chất lượng xếp hạng trong 10 vị trí đầu tiên.
- **Recall@100:** Đo tỷ lệ tài liệu liên quan được tìm thấy trong 100 vị trí đầu tiên.
- **MRR:** Đánh giá thứ hạng của tài liệu liên quan đầu tiên.
- **MAP:** Đánh giá chất lượng xếp hạng của các tài liệu liên quan.

Các chỉ số được tính cho từng truy vấn, sau đó lấy trung bình trên toàn bộ tập test.


In [34]:
from chi_so import (
    ndcg_at_k,
    recall_at_k,
    reciprocal_rank,
    average_precision
)

In [35]:
query_id = next(iter(ket_qua_bm25))

thu_hang = list(ket_qua_bm25[query_id].keys())
nhan_lien_quan = qrels[query_id]

print("Query ID:", query_id)
print("nDCG@10:", ndcg_at_k(thu_hang, nhan_lien_quan, 10))
print("Recall@100:", recall_at_k(thu_hang, nhan_lien_quan, 100))
print("RR:", reciprocal_rank(thu_hang, nhan_lien_quan))
print("AP:", average_precision(thu_hang, nhan_lien_quan))

Query ID: 1
nDCG@10: 0.0
Recall@100: 0.0
RR: 0.0
AP: 0.0


In [36]:
tong_ndcg = 0.0
tong_recall = 0.0
tong_rr = 0.0
tong_ap = 0.0

for query_id, ket_qua in ket_qua_bm25.items():
    thu_hang = list(ket_qua.keys())
    nhan_lien_quan = qrels[query_id]

    tong_ndcg += ndcg_at_k(thu_hang, nhan_lien_quan, 10)
    tong_recall += recall_at_k(thu_hang, nhan_lien_quan, 100)
    tong_rr += reciprocal_rank(thu_hang, nhan_lien_quan)
    tong_ap += average_precision(thu_hang, nhan_lien_quan)

so_truy_van = len(ket_qua_bm25)

print("=== KẾT QUẢ ĐÁNH GIÁ BM25 TRÊN SCIFACT ===")
print("nDCG@10:", round(tong_ndcg / so_truy_van, 4))
print("Recall@100:", round(tong_recall / so_truy_van, 4))
print("MRR@100:", round(tong_rr / so_truy_van, 4))
print("MAP@100:", round(tong_ap / so_truy_van, 4))

=== KẾT QUẢ ĐÁNH GIÁ BM25 TRÊN SCIFACT ===
nDCG@10: 0.5723
Recall@100: 0.7969
MRR@100: 0.5465
MAP@100: 0.5379



### 5.1. Kết quả đánh giá BM25 trên SciFact

Hệ thống BM25 tự cài đặt được đánh giá trên 300 truy vấn thuộc tập test SciFact, sử dụng cấu hình `k1 = 1.2`, `b = 0.75` và truy xuất tối đa 100 tài liệu cho mỗi truy vấn.

| Chỉ số | Kết quả |
|---|---:|
| nDCG@10 | 0.5723 |
| Recall@100 | 0.7969 |
| MRR@100 | 0.5465 |
| MAP@100 | 0.5379 |

**Nhận xét:**

- Hệ thống đạt nDCG@10 bằng 0.5723, phản ánh khả năng đưa tài liệu liên quan lên các vị trí đầu danh sách.
- Recall@100 bằng 0.7969, cho thấy hệ thống tìm được trung bình khoảng 79,69% tài liệu liên quan trong 100 kết quả đầu tiên.
- MRR@100 bằng 0.5465 và MAP@100 bằng 0.5379, phản ánh hiệu quả truy xuất và xếp hạng các tài liệu liên quan.
- Một số truy vấn không tìm được tài liệu liên quan, chẳng hạn truy vấn đầu tiên có cả bốn chỉ số bằng 0.

Các kết quả trên được sử dụng làm baseline để so sánh với Pyserini BM25 ở bước tiếp theo.


|